In [1]:
import os
import h5py
import joblib
import logging
from pathlib import Path
import numpy as np
import numpy.typing as npt
import pandas as pd
from xgboost import XGBClassifier
from keras.models import load_model
from tqdm import tqdm
from src.module.utils import (
    load_config,
    select_strategy,
    get_peak_trough,
    process_meanbp,
    load_npy,
    save_npy,
)

In [2]:
def save_lead_time_per_case(size, seg_path, event_path, config):
    for i in tqdm(range(size), ncols=75):
        segments = load_npy(seg_path[i])
        events = load_npy(event_path[i])

        lead_time_per_case = []
        for seg in segments:
            for s, e in events:
                interval = config["fs"] * 60 * 5
                lead_time_10to15 = [e - interval, e]
                lead_time_5to10 = [lead_time_10to15[0] - interval, lead_time_10to15[0]]
                lead_time_0to5 = [lead_time_5to10[0] - interval, lead_time_5to10[0]]

                is_10to15 = np.all(
                    (seg >= lead_time_10to15[0]) & (seg < lead_time_10to15[1])
                )
                is_5to10 = np.all(
                    (seg >= lead_time_5to10[0]) & (seg < lead_time_5to10[1])
                )
                is_0to5 = np.all((seg >= lead_time_0to5[0]) & (seg < lead_time_0to5[1]))

                # 만약 새그먼트가 첫번째 이벤트에서 어느 한 쪽에 속한다면
                if is_10to15 or is_5to10 or is_0to5:
                    break

            if is_0to5:
                lead_time_per_case.append(0)
            elif is_5to10:
                lead_time_per_case.append(1)
            elif is_10to15:
                lead_time_per_case.append(2)
            else:
                lead_time_per_case.append(-1)

        # 새그먼트 별로 lead-time 라벨 저장
        lead_time_per_case = np.array(lead_time_per_case)
        # print(f"{str(seg_path[i])[:-13]}_lead_times.npy")
        save_npy(f"{str(seg_path[i])[:-13]}_lead_times.npy", lead_time_per_case)

In [3]:
def load_path(src_path, strategy="hypophetversion2"):
    pos_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_pos_*.npy"))
    )
    neg_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_neg_*.npy"))
    )
    return pos_path, neg_path


def load_dataset(pos_path, neg_path) -> tuple[npt.NDArray, npt.NDArray]:
    pos_cid = load_npy(pos_path[0])
    neg_cid = load_npy(neg_path[0])
    # env['pos_path'] : pos_lt
    pos_lt = load_npy(pos_path[1])
    neg_lt = np.full(len(neg_cid), -1)
    pos_seg = load_npy(pos_path[2])
    neg_seg = load_npy(neg_path[1])
    pos_ts = load_npy(pos_path[3])
    neg_ts = load_npy(neg_path[2])

    pos_y, neg_y = np.ones(len(pos_seg)), np.zeros(len(neg_seg))
    X, y = np.vstack([pos_seg, neg_seg]), np.hstack([pos_y, neg_y])
    cid = np.hstack([pos_cid, neg_cid])
    ts = np.vstack([pos_ts, neg_ts])
    lt = np.hstack([pos_lt, neg_lt])
    logging.info(f"load_dataset(pos/neg): {y.size}({pos_y.size}/{neg_y.size})")
    print(X.shape, y.shape, cid.shape, ts.shape, lt.shape)
    return cid, ts, lt, X, y

In [4]:
def extract_lastmbp(arr, config):
    mbp = process_meanbp(arr.reshape(-1), config)
    mbp = np.unique(mbp[np.where(mbp != 0)])
    return mbp[-1]


def get_min_size(x, p, t):
    if x[p].size < x[t].size:
        min_size = x[p].size
    else:
        min_size = x[t].size
    return min_size


def summary_stat(arr):
    x = arr.reshape(-1)
    x_mean, x_std, x_median, x_min, x_max = (
        np.mean(x),
        np.std(x),
        np.median(x),
        np.min(x),
        np.max(x),
    )
    return x_mean, x_std, x_median, x_min, x_max


def slope_intercept(t, y):
    t = np.asarray(t, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    tm, ym = t.mean(), y.mean()
    dt = t - tm
    denom = (dt * dt).sum()
    if denom == 0:
        return np.nan, np.nan
    slope = (dt * (y - ym)).sum() / denom
    return slope, ym - (slope * tm)


def extract_feature(arr, config):
    feature = []
    p, t = get_peak_trough(arr, config["detection"])
    min_size = get_min_size(arr, p, t)
    mbp = process_meanbp(arr.reshape(-1), config)
    sbp, dbp, pp = arr[p], arr[t], arr[p][:min_size] - arr[t][:min_size]
    mbp = np.unique(mbp[np.where(mbp != 0)])

    sbp_stat = summary_stat(sbp)
    dbp_stat = summary_stat(dbp)
    pp_stat = summary_stat(pp)
    mbp_stat = summary_stat(mbp)
    trend_per_beat, _ = slope_intercept(np.arange(mbp.size), mbp)
    trend_per_sec = trend_per_beat * (mbp.size / 30.0)
    half = mbp.size // 2
    slope_per_beat, _ = slope_intercept(np.arange(mbp.size)[half:], mbp[half:])
    slope_per_sec = slope_per_beat * (mbp.size / 30.0)
    last_mbp = mbp[-1]

    feature.extend(
        [
            *sbp_stat,
            *dbp_stat,
            *pp_stat,
            *mbp_stat,
            trend_per_sec,
            slope_per_sec,
            last_mbp,
        ]
    )
    return feature


def scale_input(input) -> npt.NDArray:
    return input.astype(np.float32) / 200

In [5]:
def score_sampled_auroc(idx_by_pat, true_y, prob_y):
    pat_ids = np.array(list(idx_by_pat.keys()))
    sampled = np.random.choice(pat_ids, size=len(pat_ids), replace=True)
    idx = np.concatenate([idx_by_pat[p] for p in sampled])
    sampled_y = true_y[idx]
    sampled_mbp = prob_y[idx]
    auroc = roc_auc_score(sampled_y, sampled_mbp)
    return auroc


def eval_sampled_clf(idx_by_pat, true_y, prob_y, threshold):
    pat_ids = np.array(list(idx_by_pat.keys()))
    sampled = np.random.choice(pat_ids, size=len(pat_ids), replace=True)
    idx = np.concatenate([idx_by_pat[p] for p in sampled])
    sampled_true_y = true_y[idx]
    sampled_prob_y = prob_y[idx]
    pred = (sampled_prob_y >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(sampled_true_y, pred).ravel()
    sen, spe = tp / (tp + fn), tn / (tn + fp)
    return sen, spe

In [6]:
def patient_level_bootstrap(n, cid_grp: list, true_y: list, prob_y: list, thrs: list):
    idx_by_pat = [{p: np.where(cid == p)[0] for p in np.unique(cid)} for cid in cid_grp]
    np.random.seed(config["seed"])  # 42

    auroc_list = [[], [], []]
    delta_list = [[], [], []]
    sen_list = [[[], [], []], [[], [], []], [[], [], []]]
    spe_list = [[], [], []]
    # auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
    # 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
    # delta_list1, delta_list2, delta_list3 = [], [], []
    # sen_0to5_list, sen_5to10_list, sen_10to15_list = [], [], []
    # spe_0to5_list, spe_5to10_list, spe_10to15_list = [], [], []

    for _ in tqdm(range(n), ncols=75):
        for i in range(len(cid_grp)):
            auroc = score_sampled_auroc(idx_by_pat[i], true_y[i], prob_y[i])
            auroc_list[i].append(auroc)
            # for j in range(len(thrs)):
            #     sen, spe = eval_sampled_clf(
            #         idx_by_pat[i], true_y[i], prob_y[i] * -1, thrs[j]
            #     )
            #     sen_list[i][j].append(sen)
            # spe_list[i].append(spe)

    # delta_list1 = np.diff(auroc_list[1], auroc_list[0])
    # delta_list2 = np.diff(auroc_list[2], auroc_list[1])
    # delta_list3 = np.diff(auroc_list[2], auroc_list[0])

    print("95% CI (without gray)")
    print(
        f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_list[0], 2.5):.3f}-{np.percentile(auroc_list[0], 97.5):.3f})"
    )
    print(
        f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
    )
    print(
        f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_list[1], 2.5):.3f}-{np.percentile(auroc_list[1], 97.5):.3f})"
    )
    print(
        f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
    )
    print(
        f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_list[2], 2.5):.3f}-{np.percentile(auroc_list[2], 97.5):.3f})"
    )
    print(
        f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
    )

    print("Spe80% (train):")
    print(
        f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_list[0][0], 2.5):.3f}-{np.percentile(sen_list[0][0], 97.5):.3f})"
    )
    print(
        f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_list[1][0], 2.5):.3f}-{np.percentile(sen_list[1][0], 97.5):.3f})"
    )
    print(
        f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_list[2][0], 2.5):.3f}-{np.percentile(sen_list[2][0], 97.5):.3f})"
    )
    print("-----" * 5)
    print(
        f"- Spe: ({np.percentile(spe_list[0], 2.5):.3f}-{np.percentile(spe_list[0], 97.5):.3f})"
    )

    print("Spe90% (train):")
    print(
        f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_list[0][1], 2.5):.3f}-{np.percentile(sen_list[0][1], 97.5):.3f})"
    )
    print(
        f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_list[1][1], 2.5):.3f}-{np.percentile(sen_list[1][1], 97.5):.3f})"
    )
    print(
        f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_list[2][1], 2.5):.3f}-{np.percentile(sen_list[2][1], 97.5):.3f})"
    )
    print("-----" * 5)
    print(
        f"- Spe: ({np.percentile(spe_list[1], 2.5):.3f}-{np.percentile(spe_list[1], 97.5):.3f})"
    )

    print("Spe95% (train):")
    print(
        f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_list[0][2], 2.5):.3f}-{np.percentile(sen_list[0][2], 97.5):.3f})"
    )
    print(
        f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_list[1][2], 2.5):.3f}-{np.percentile(sen_list[1][2], 97.5):.3f})"
    )
    print(
        f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_list[2][2], 2.5):.3f}-{np.percentile(sen_list[2][2], 97.5):.3f})"
    )
    print("-----" * 5)
    print(
        f"- Spe: ({np.percentile(spe_list[2], 2.5):.3f}-{np.percentile(spe_list[2], 97.5):.3f})"
    )

# Save lead-time

04_load_segment_data_mover.py 를 참고해서 데이터를 가져온다.

lead-time:
- 0: 0min ≤ t < 5min
- 1: 5min ≤ t < 10min
- 2: 10min ≤ t < 15min

In [3]:
config = load_config("../../config/conf.yaml")
strategy = select_strategy("hypophetversion2")
data_path = Path(config["data_path"]).expanduser()
ref_path = data_path / "mover" / "03.label"
src_path = data_path / "mover" / "04.segment"

HypophetVersion2 strategy selected.


In [4]:
pos_event_path = sorted(ref_path.rglob(f"*{strategy.name}*_pred_pos_events.npy"))
pos_event_path = [x for x in pos_event_path if not x.name.startswith("._")]
neg_event_path = sorted(ref_path.rglob(f"*{strategy.name}*_neg_events.npy"))
neg_event_path = [x for x in neg_event_path if not x.name.startswith("._")]
print(len(pos_event_path))
print(len(neg_event_path))

1388
1388


In [5]:
pos_seg_path = sorted(src_path.rglob(f"*{strategy.name}*_pred_pos_segments.npy"))
pos_seg_path = [x for x in pos_seg_path if not x.name.startswith("._")]
neg_seg_path = sorted(src_path.rglob(f"*{strategy.name}*_neg_segments.npy"))
neg_seg_path = [x for x in neg_seg_path if not x.name.startswith("._")]
print(len(pos_seg_path))
print(len(neg_seg_path))

1388
1388


In [6]:
# sanity check: 모든 양성 event의 간격이 15분인가? No.
# layer를 쌓듯이 라벨을 쌓았다.
# 그러므로 만약 event가 [s, e] 라면, e - 5분부터 시작하면 될 듯.
# all_events = np.vstack([load_npy(e) for e in pos_event_path if len(load_npy(e)) != 0])
# print(all_events.shape)

In [7]:
# 1. 각 케이스 별로 순회한다.
# 2. 케이스에 대응하는 event, segment를 불러온다.
# 3. event 별로 lead-time을 분할한다.
# 4. segment가 어느 lead-time에 속하는지 라벨링한다. (0: 0~5, 1: 5~10, 2: 10~15)

In [8]:
# events = load_npy(pos_event_path[10])
# segments = load_npy(pos_seg_path[10])
# print(events.shape, segments.shape)

In [10]:
save_lead_time_per_case(len(pos_seg_path), pos_seg_path, pos_event_path, config)

100%|██████████████████████████████████| 1388/1388 [01:17<00:00, 17.88it/s]


In [ ]:
# neg lead time이라는 건 없지 않아?
# save_lead_time_per_case(len(neg_seg_path), neg_seg_path, neg_event_path, config)

100%|██████████████████████████████████| 1388/1388 [01:52<00:00, 12.30it/s]


In [12]:
# for i in range(len(pos_event_path))[:3]:
#     segments = load_npy(pos_seg_path[i])
#     events = load_npy(pos_event_path[i])

#     lead_time_per_case = []
#     for seg in segments:
#         for s, e in events:
#             interval = config["fs"] * 60 * 5
#             lead_time_10to15 = [e - interval, e]
#             lead_time_5to10 = [lead_time_10to15[0] - interval, lead_time_10to15[0]]
#             lead_time_0to5 = [lead_time_5to10[0] - interval, lead_time_5to10[0]]

#             is_10to15 = np.all(
#                 (seg >= lead_time_10to15[0]) & (seg < lead_time_10to15[1])
#             )
#             is_5to10 = np.all((seg >= lead_time_5to10[0]) & (seg < lead_time_5to10[1]))
#             is_0to5 = np.all((seg >= lead_time_0to5[0]) & (seg < lead_time_0to5[1]))

#             # 만약 새그먼트가 첫번째 이벤트에서 어느 한 쪽에 속한다면
#             if is_10to15 or is_5to10 or is_0to5:
#                 break

#         if is_0to5:
#             lead_time_per_case.append(0)
#         elif is_5to10:
#             lead_time_per_case.append(1)
#         elif is_10to15:
#             lead_time_per_case.append(2)
#         else:
#             lead_time_per_case.append(-1)

#     # 새그먼트 별로 lead-time 라벨 저장
#     lead_time_per_case = np.array(lead_time_per_case)
#     save_npy(str(pos_seg_path[i])[:-13] / "_lead_times.npy", lead_time_per_case)

# load lead-times

In [20]:
pos_lt_path = sorted(src_path.rglob(f"*{strategy.name}*_pred_pos_lead_times.npy"))
pos_lt_path = [x for x in pos_lt_path if not x.name.startswith("._")]
print(len(pos_lt_path))

1388


In [23]:
pos_lt = np.hstack([load_npy(lt) for lt in pos_lt_path if len(load_npy(lt)) != 0])

In [25]:
np.unique(pos_lt, return_counts=True)

(array([0, 1, 2]), array([2635, 3021, 4046]))

# Eval

04에서 산출된 lead_times를 이용해서 05를 실행, pos_lt.npy를 생성한다.

#### 결과 산출 방법

1. 리드타임 별 데이터셋을 음성 데이터셋과 비교하여 성능 결과 산출 (spe 80/90/95% 고정; 부트스트랩 포함)

2. 리드타임 별 auroc 에 bootstrap 으로 CI 추가

3. violin-plot을 이용해 세 가지 리드타임 데이터셋 결과 확률값 비교

사용지표: auroc, ∆AUROC, spe, sen

lead-time:
- 0: 0min ≤ t < 5min
- 1: 5min ≤ t < 10min
- 2: 10min ≤ t < 15min

In [7]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
)

In [8]:
config = load_config("../../config/conf.yaml")
strategy = select_strategy("hypophetversion2")
data_path = Path(config["data_path"]).expanduser() / "mover" / "05.dataset_new"
model_path = Path(config["model_path"]).expanduser() / "revision_seed42"

HypophetVersion2 strategy selected.


In [9]:
pos_path, neg_path = load_path(data_path)

In [10]:
cid, ts, lt, X, y = load_dataset(pos_path, neg_path)
print(np.unique(cid).shape, cid.shape, X.shape, y.shape, sum(y == 1), sum(y == 0))
print(sum(lt == -1), sum(lt == 0), sum(lt == 1), sum(lt == 2))

(25793, 3000) (25793,) (25793,) (25793, 3000) (25793,)
(649,) (25793,) (25793, 3000) (25793,) 9295 16498
16498 2525 2894 3876


In [11]:
cid_0to5, X_0to5, y_0to5 = (
    cid[np.where((lt == -1) | (lt == 0))],
    X[np.where((lt == -1) | (lt == 0))],
    y[np.where((lt == -1) | (lt == 0))],
)
cid_5to10, X_5to10, y_5to10 = (
    cid[np.where((lt == -1) | (lt == 1))],
    X[np.where((lt == -1) | (lt == 1))],
    y[np.where((lt == -1) | (lt == 1))],
)
cid_10to15, X_10to15, y_10to15 = (
    cid[np.where((lt == -1) | (lt == 2))],
    X[np.where((lt == -1) | (lt == 2))],
    y[np.where((lt == -1) | (lt == 2))],
)
print(X_0to5.shape, X_5to10.shape, X_10to15.shape)

(19023, 3000) (19392, 3000) (20374, 3000)


In [12]:
X_feat_0to5 = np.array([extract_feature(x, config) for x in tqdm(X_0to5, ncols=75)])
print(X_feat_0to5.shape)
X_feat_5to10 = np.array([extract_feature(x, config) for x in tqdm(X_5to10, ncols=75)])
print(X_feat_5to10.shape)
X_feat_10to15 = np.array([extract_feature(x, config) for x in tqdm(X_10to15, ncols=75)])
print(X_feat_10to15.shape)

100%|██████████████████████████████| 19023/19023 [00:06<00:00, 3006.31it/s]


(19023, 23)


100%|██████████████████████████████| 19392/19392 [00:06<00:00, 2980.82it/s]


(19392, 23)


100%|██████████████████████████████| 20374/20374 [00:06<00:00, 3044.11it/s]

(20374, 23)


In [13]:
X_0to5_s = scale_input(X_0to5)
X_5to10_s = scale_input(X_5to10)
X_10to15_s = scale_input(X_10to15)

# MBP Baseline

In [44]:
lastmbp_0to5 = np.array([extract_lastmbp(x, config) for x in tqdm(X_0to5, ncols=75)])
print(lastmbp_0to5.shape)
lastmbp_5to10 = np.array([extract_lastmbp(x, config) for x in tqdm(X_5to10, ncols=75)])
print(lastmbp_5to10.shape)
lastmbp_10to15 = np.array(
    [extract_lastmbp(x, config) for x in tqdm(X_10to15, ncols=75)]
)
print(lastmbp_10to15.shape)

100%|██████████████████████████████| 19023/19023 [00:03<00:00, 5038.71it/s]


(19023,)


100%|██████████████████████████████| 19392/19392 [00:03<00:00, 5142.32it/s]


(19392,)


100%|██████████████████████████████| 20374/20374 [00:03<00:00, 5172.08it/s]

(20374,)


In [45]:
print(f"AUROC(0min ≤ t < 5min): {roc_auc_score(y_0to5, -lastmbp_0to5):.3f}")
print(f"AUROC(5min ≤ t < 10min): {roc_auc_score(y_5to10, -lastmbp_5to10):.3f}")
print(f"AUROC(10min ≤ t < 15min): {roc_auc_score(y_10to15, -lastmbp_10to15):.3f}")

AUROC(0min ≤ t < 5min): 0.626
AUROC(5min ≤ t < 10min): 0.650
AUROC(10min ≤ t < 15min): 0.779


In [46]:
print("Specificity(train): 0.8")
pred_0to5 = (lastmbp_0to5 < 80.36).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (lastmbp_5to10 < 80.36).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (lastmbp_10to15 < 80.36).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.8
Sen(0 ≤ t < 5): 0.418
Sen(5 ≤ t < 10): 0.441
Sen(10 ≤ t < 15): 0.665
-------------------------
Spe: 0.775


In [47]:
print("Specificity(train): 0.9")
pred_0to5 = (lastmbp_0to5 < 76.38).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (lastmbp_5to10 < 76.38).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (lastmbp_10to15 < 76.38).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.9
Sen(0 ≤ t < 5): 0.244
Sen(5 ≤ t < 10): 0.290
Sen(10 ≤ t < 15): 0.491
-------------------------
Spe: 0.887


In [48]:
print("Specificity(train): 0.95")
pred_0to5 = (lastmbp_0to5 < 73.68).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (lastmbp_5to10 < 73.68).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (lastmbp_10to15 < 73.68).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.95
Sen(0 ≤ t < 5): 0.141
Sen(5 ≤ t < 10): 0.175
Sen(10 ≤ t < 15): 0.350
-------------------------
Spe: 0.946


In [57]:
pat_ids_0to5 = np.unique(cid_0to5)
pat_ids_5to10 = np.unique(cid_5to10)
pat_ids_10to15 = np.unique(cid_10to15)
idx_by_pat_0to5 = {p: np.where(cid_0to5 == p)[0] for p in pat_ids_0to5}
idx_by_pat_5to10 = {p: np.where(cid_5to10 == p)[0] for p in pat_ids_5to10}
idx_by_pat_10to15 = {p: np.where(cid_10to15 == p)[0] for p in pat_ids_10to15}
np.random.seed(config["seed"])  # 42

auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
# 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
delta_list1, delta_list2, delta_list3 = [], [], []
sen_0to5_thr80_list, sen_5to10_thr80_list, sen_10to15_thr80_list = [], [], []
sen_0to5_thr90_list, sen_5to10_thr90_list, sen_10to15_thr90_list = [], [], []
sen_0to5_thr95_list, sen_5to10_thr95_list, sen_10to15_thr95_list = [], [], []
spe_thr80_list, spe_thr90_list, spe_thr95_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_0to5 = score_sampled_auroc(idx_by_pat_0to5, y_0to5, -lastmbp_0to5)
    auroc_5to10 = score_sampled_auroc(idx_by_pat_5to10, y_5to10, -lastmbp_5to10)
    auroc_10to15 = score_sampled_auroc(idx_by_pat_10to15, y_10to15, -lastmbp_10to15)
    auroc_0to5_list.append(auroc_0to5)
    auroc_5to10_list.append(auroc_5to10)
    auroc_10to15_list.append(auroc_10to15)
    delta_list1.append(auroc_0to5 - auroc_5to10)
    delta_list2.append(auroc_5to10 - auroc_10to15)
    delta_list3.append(auroc_10to15 - auroc_0to5)

    # Spe (train): 80%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, lastmbp_0to5, 80.36)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, lastmbp_5to10, 80.36
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, lastmbp_10to15, 80.36
    )
    sen_0to5_thr80_list.append(sen_0to5)
    sen_5to10_thr80_list.append(sen_5to10)
    sen_10to15_thr80_list.append(sen_10to15)
    spe_thr80_list.append(spe_0to5)

    # Spe (train): 90%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, lastmbp_0to5, 76.38)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, lastmbp_5to10, 76.38
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, lastmbp_10to15, 76.38
    )
    sen_0to5_thr90_list.append(sen_0to5)
    sen_5to10_thr90_list.append(sen_5to10)
    sen_10to15_thr90_list.append(sen_10to15)
    spe_thr90_list.append(spe_0to5)

    # Spe (train): 95%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, lastmbp_0to5, 73.68)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, lastmbp_5to10, 73.68
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, lastmbp_10to15, 73.68
    )
    sen_0to5_thr95_list.append(sen_0to5)
    sen_5to10_thr95_list.append(sen_5to10)
    sen_10to15_thr95_list.append(sen_10to15)
    spe_thr95_list.append(spe_0to5)

100%|██████████████████████████████████| 2000/2000 [01:06<00:00, 30.28it/s]


In [58]:
print("95% CI (without gray)")
print(
    f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_0to5_list, 2.5):.3f}-{np.percentile(auroc_0to5_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
)
print(
    f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_5to10_list, 2.5):.3f}-{np.percentile(auroc_5to10_list, 97.5):.3f})"
)
print(
    f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
)
print(
    f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_10to15_list, 2.5):.3f}-{np.percentile(auroc_10to15_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
)

print("Spe80% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr80_list, 2.5):.3f}-{np.percentile(sen_0to5_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr80_list, 2.5):.3f}-{np.percentile(sen_5to10_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr80_list, 2.5):.3f}-{np.percentile(sen_10to15_thr80_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr80_list, 2.5):.3f}-{np.percentile(spe_thr80_list, 97.5):.3f})"
)

print("Spe90% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr90_list, 2.5):.3f}-{np.percentile(sen_0to5_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr90_list, 2.5):.3f}-{np.percentile(sen_5to10_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr90_list, 2.5):.3f}-{np.percentile(sen_10to15_thr90_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr90_list, 2.5):.3f}-{np.percentile(spe_thr90_list, 97.5):.3f})"
)

print("Spe95% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr95_list, 2.5):.3f}-{np.percentile(sen_0to5_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr95_list, 2.5):.3f}-{np.percentile(sen_5to10_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr95_list, 2.5):.3f}-{np.percentile(sen_10to15_thr95_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr95_list, 2.5):.3f}-{np.percentile(spe_thr95_list, 97.5):.3f})"
)

95% CI (without gray)
AUROC(0 ≤ t < 5): (0.595-0.657)
∆AUROC (0to5 vs. 5to10): (-0.066-0.016)
AUROC(5 ≤ t < 10): (0.621-0.678)
∆AUROC (5to10 vs. 10to15): (-0.165--0.095)
AUROC(10 ≤ t < 15): (0.759-0.799)
∆AUROC (0to5 vs. 10to15): (0.116-0.191)
Spe80% (train):
- Sen(0 ≤ t < 5):: (0.374-0.463)
- Sen(5 ≤ t < 10):: (0.400-0.483)
- Sen(10 ≤ t < 15):: (0.634-0.694)
-------------------------
- Spe: (0.751-0.798)
Spe90% (train):
- Sen(0 ≤ t < 5):: (0.206-0.283)
- Sen(5 ≤ t < 10):: (0.254-0.329)
- Sen(10 ≤ t < 15):: (0.460-0.523)
-------------------------
- Spe: (0.873-0.901)
Spe95% (train):
- Sen(0 ≤ t < 5):: (0.110-0.173)
- Sen(5 ≤ t < 10):: (0.146-0.205)
- Sen(10 ≤ t < 15):: (0.320-0.379)
-------------------------
- Spe: (0.937-0.954)


# LR

In [14]:
lrpipe = joblib.load(model_path / "lrpipe_baseline.joblib")

In [15]:
prob_y_0to5 = lrpipe["model"].predict_proba(X_feat_0to5)[:, 1]
prob_y_5to10 = lrpipe["model"].predict_proba(X_feat_5to10)[:, 1]
prob_y_10to15 = lrpipe["model"].predict_proba(X_feat_10to15)[:, 1]

In [16]:
print(f"AUROC(0min ≤ t < 5min): {roc_auc_score(y_0to5, prob_y_0to5):.3f}")
print(f"AUROC(5min ≤ t < 10min): {roc_auc_score(y_5to10, prob_y_5to10):.3f}")
print(f"AUROC(10min ≤ t < 15min): {roc_auc_score(y_10to15, prob_y_10to15):.3f}")

AUROC(0min ≤ t < 5min): 0.672
AUROC(5min ≤ t < 10min): 0.698
AUROC(10min ≤ t < 15min): 0.816


In [17]:
print("Specificity(train): 0.8")
pred_0to5 = (prob_y_0to5 >= 0.61).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.61).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.61).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.8
Sen(0 ≤ t < 5): 0.561
Sen(5 ≤ t < 10): 0.597
Sen(10 ≤ t < 15): 0.795
-------------------------
Spe: 0.701


In [18]:
print("Specificity(train): 0.9")
pred_0to5 = (prob_y_0to5 >= 0.69).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.69).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.69).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.9
Sen(0 ≤ t < 5): 0.383
Sen(5 ≤ t < 10): 0.434
Sen(10 ≤ t < 15): 0.676
-------------------------
Spe: 0.811


In [19]:
print("Specificity(train): 0.95")
pred_0to5 = (prob_y_0to5 >= 0.75).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.75).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.75).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.95
Sen(0 ≤ t < 5): 0.252
Sen(5 ≤ t < 10): 0.319
Sen(10 ≤ t < 15): 0.517
-------------------------
Spe: 0.890


In [20]:
pat_ids_0to5 = np.unique(cid_0to5)
pat_ids_5to10 = np.unique(cid_5to10)
pat_ids_10to15 = np.unique(cid_10to15)
idx_by_pat_0to5 = {p: np.where(cid_0to5 == p)[0] for p in pat_ids_0to5}
idx_by_pat_5to10 = {p: np.where(cid_5to10 == p)[0] for p in pat_ids_5to10}
idx_by_pat_10to15 = {p: np.where(cid_10to15 == p)[0] for p in pat_ids_10to15}
np.random.seed(config["seed"])  # 42

auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
# 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
delta_list1, delta_list2, delta_list3 = [], [], []
sen_0to5_thr80_list, sen_5to10_thr80_list, sen_10to15_thr80_list = [], [], []
sen_0to5_thr90_list, sen_5to10_thr90_list, sen_10to15_thr90_list = [], [], []
sen_0to5_thr95_list, sen_5to10_thr95_list, sen_10to15_thr95_list = [], [], []
spe_thr80_list, spe_thr90_list, spe_thr95_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_0to5 = score_sampled_auroc(idx_by_pat_0to5, y_0to5, prob_y_0to5)
    auroc_5to10 = score_sampled_auroc(idx_by_pat_5to10, y_5to10, prob_y_5to10)
    auroc_10to15 = score_sampled_auroc(idx_by_pat_10to15, y_10to15, prob_y_10to15)
    auroc_0to5_list.append(auroc_0to5)
    auroc_5to10_list.append(auroc_5to10)
    auroc_10to15_list.append(auroc_10to15)
    delta_list1.append(auroc_0to5 - auroc_5to10)
    delta_list2.append(auroc_5to10 - auroc_10to15)
    delta_list3.append(auroc_10to15 - auroc_0to5)

    # Spe (train): 80%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.61)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.61
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.61
    )
    sen_0to5_thr80_list.append(sen_0to5)
    sen_5to10_thr80_list.append(sen_5to10)
    sen_10to15_thr80_list.append(sen_10to15)
    spe_thr80_list.append(spe_0to5)

    # Spe (train): 90%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.69)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.69
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.69
    )
    sen_0to5_thr90_list.append(sen_0to5)
    sen_5to10_thr90_list.append(sen_5to10)
    sen_10to15_thr90_list.append(sen_10to15)
    spe_thr90_list.append(spe_0to5)

    # Spe (train): 95%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.75)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.75
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.75
    )
    sen_0to5_thr95_list.append(sen_0to5)
    sen_5to10_thr95_list.append(sen_5to10)
    sen_10to15_thr95_list.append(sen_10to15)
    spe_thr95_list.append(spe_0to5)

100%|██████████████████████████████████| 2000/2000 [01:08<00:00, 29.14it/s]


In [21]:
print("95% CI (without gray)")
print(
    f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_0to5_list, 2.5):.3f}-{np.percentile(auroc_0to5_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
)
print(
    f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_5to10_list, 2.5):.3f}-{np.percentile(auroc_5to10_list, 97.5):.3f})"
)
print(
    f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
)
print(
    f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_10to15_list, 2.5):.3f}-{np.percentile(auroc_10to15_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
)

print("Spe80% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr80_list, 2.5):.3f}-{np.percentile(sen_0to5_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr80_list, 2.5):.3f}-{np.percentile(sen_5to10_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr80_list, 2.5):.3f}-{np.percentile(sen_10to15_thr80_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr80_list, 2.5):.3f}-{np.percentile(spe_thr80_list, 97.5):.3f})"
)

print("Spe90% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr90_list, 2.5):.3f}-{np.percentile(sen_0to5_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr90_list, 2.5):.3f}-{np.percentile(sen_5to10_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr90_list, 2.5):.3f}-{np.percentile(sen_10to15_thr90_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr90_list, 2.5):.3f}-{np.percentile(spe_thr90_list, 97.5):.3f})"
)

print("Spe95% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr95_list, 2.5):.3f}-{np.percentile(sen_0to5_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr95_list, 2.5):.3f}-{np.percentile(sen_5to10_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr95_list, 2.5):.3f}-{np.percentile(sen_10to15_thr95_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr95_list, 2.5):.3f}-{np.percentile(spe_thr95_list, 97.5):.3f})"
)

95% CI (without gray)
AUROC(0 ≤ t < 5): (0.644-0.701)
∆AUROC (0to5 vs. 5to10): (-0.064-0.011)
AUROC(5 ≤ t < 10): (0.672-0.722)
∆AUROC (5to10 vs. 10to15): (-0.150--0.085)
AUROC(10 ≤ t < 15): (0.797-0.833)
∆AUROC (0to5 vs. 10to15): (0.109-0.177)
Spe80% (train):
- Sen(0 ≤ t < 5):: (0.518-0.604)
- Sen(5 ≤ t < 10):: (0.555-0.637)
- Sen(10 ≤ t < 15):: (0.769-0.819)
-------------------------
- Spe: (0.673-0.727)
Spe90% (train):
- Sen(0 ≤ t < 5):: (0.339-0.426)
- Sen(5 ≤ t < 10):: (0.391-0.475)
- Sen(10 ≤ t < 15):: (0.646-0.704)
-------------------------
- Spe: (0.789-0.832)
Spe95% (train):
- Sen(0 ≤ t < 5):: (0.215-0.292)
- Sen(5 ≤ t < 10):: (0.280-0.361)
- Sen(10 ≤ t < 15):: (0.483-0.550)
-------------------------
- Spe: (0.874-0.905)


# XGB

In [23]:
loaded_xgb = XGBClassifier()
loaded_xgb.load_model(model_path / "xgb_baseline.json")

In [24]:
prob_y_0to5 = loaded_xgb.predict_proba(X_feat_0to5)[:, 1]
prob_y_5to10 = loaded_xgb.predict_proba(X_feat_5to10)[:, 1]
prob_y_10to15 = loaded_xgb.predict_proba(X_feat_10to15)[:, 1]

In [25]:
print(f"AUROC(0min ≤ t < 5min): {roc_auc_score(y_0to5, prob_y_0to5):.3f}")
print(f"AUROC(5min ≤ t < 10min): {roc_auc_score(y_5to10, prob_y_5to10):.3f}")
print(f"AUROC(10min ≤ t < 15min): {roc_auc_score(y_10to15, prob_y_10to15):.3f}")

AUROC(0min ≤ t < 5min): 0.672
AUROC(5min ≤ t < 10min): 0.697
AUROC(10min ≤ t < 15min): 0.818


In [26]:
print("Specificity(train): 0.8")
pred_0to5 = (prob_y_0to5 >= 0.59).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.59).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.59).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.8
Sen(0 ≤ t < 5): 0.509
Sen(5 ≤ t < 10): 0.562
Sen(10 ≤ t < 15): 0.770
-------------------------
Spe: 0.711


In [27]:
print("Specificity(train): 0.9")
pred_0to5 = (prob_y_0to5 >= 0.72).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.72).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.72).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.9
Sen(0 ≤ t < 5): 0.337
Sen(5 ≤ t < 10): 0.383
Sen(10 ≤ t < 15): 0.626
-------------------------
Spe: 0.841


In [28]:
print("Specificity(train): 0.95")
pred_0to5 = (prob_y_0to5 >= 0.82).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.82).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.82).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.95
Sen(0 ≤ t < 5): 0.213
Sen(5 ≤ t < 10): 0.257
Sen(10 ≤ t < 15): 0.455
-------------------------
Spe: 0.921


In [29]:
pat_ids_0to5 = np.unique(cid_0to5)
pat_ids_5to10 = np.unique(cid_5to10)
pat_ids_10to15 = np.unique(cid_10to15)
idx_by_pat_0to5 = {p: np.where(cid_0to5 == p)[0] for p in pat_ids_0to5}
idx_by_pat_5to10 = {p: np.where(cid_5to10 == p)[0] for p in pat_ids_5to10}
idx_by_pat_10to15 = {p: np.where(cid_10to15 == p)[0] for p in pat_ids_10to15}
np.random.seed(config["seed"])  # 42

auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
# 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
delta_list1, delta_list2, delta_list3 = [], [], []
sen_0to5_thr80_list, sen_5to10_thr80_list, sen_10to15_thr80_list = [], [], []
sen_0to5_thr90_list, sen_5to10_thr90_list, sen_10to15_thr90_list = [], [], []
sen_0to5_thr95_list, sen_5to10_thr95_list, sen_10to15_thr95_list = [], [], []
spe_thr80_list, spe_thr90_list, spe_thr95_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_0to5 = score_sampled_auroc(idx_by_pat_0to5, y_0to5, prob_y_0to5)
    auroc_5to10 = score_sampled_auroc(idx_by_pat_5to10, y_5to10, prob_y_5to10)
    auroc_10to15 = score_sampled_auroc(idx_by_pat_10to15, y_10to15, prob_y_10to15)
    auroc_0to5_list.append(auroc_0to5)
    auroc_5to10_list.append(auroc_5to10)
    auroc_10to15_list.append(auroc_10to15)
    delta_list1.append(auroc_0to5 - auroc_5to10)
    delta_list2.append(auroc_5to10 - auroc_10to15)
    delta_list3.append(auroc_10to15 - auroc_0to5)

    # Spe (train): 80%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.59)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.59
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.59
    )
    sen_0to5_thr80_list.append(sen_0to5)
    sen_5to10_thr80_list.append(sen_5to10)
    sen_10to15_thr80_list.append(sen_10to15)
    spe_thr80_list.append(spe_0to5)

    # Spe (train): 90%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.72)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.72
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.72
    )
    sen_0to5_thr90_list.append(sen_0to5)
    sen_5to10_thr90_list.append(sen_5to10)
    sen_10to15_thr90_list.append(sen_10to15)
    spe_thr90_list.append(spe_0to5)

    # Spe (train): 95%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.82)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.82
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.82
    )
    sen_0to5_thr95_list.append(sen_0to5)
    sen_5to10_thr95_list.append(sen_5to10)
    sen_10to15_thr95_list.append(sen_10to15)
    spe_thr95_list.append(spe_0to5)

100%|██████████████████████████████████| 2000/2000 [01:50<00:00, 18.16it/s]


In [30]:
print("95% CI (without gray)")
print(
    f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_0to5_list, 2.5):.3f}-{np.percentile(auroc_0to5_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
)
print(
    f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_5to10_list, 2.5):.3f}-{np.percentile(auroc_5to10_list, 97.5):.3f})"
)
print(
    f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
)
print(
    f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_10to15_list, 2.5):.3f}-{np.percentile(auroc_10to15_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
)

print("Spe80% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr80_list, 2.5):.3f}-{np.percentile(sen_0to5_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr80_list, 2.5):.3f}-{np.percentile(sen_5to10_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr80_list, 2.5):.3f}-{np.percentile(sen_10to15_thr80_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr80_list, 2.5):.3f}-{np.percentile(spe_thr80_list, 97.5):.3f})"
)

print("Spe90% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr90_list, 2.5):.3f}-{np.percentile(sen_0to5_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr90_list, 2.5):.3f}-{np.percentile(sen_5to10_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr90_list, 2.5):.3f}-{np.percentile(sen_10to15_thr90_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr90_list, 2.5):.3f}-{np.percentile(spe_thr90_list, 97.5):.3f})"
)

print("Spe95% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr95_list, 2.5):.3f}-{np.percentile(sen_0to5_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr95_list, 2.5):.3f}-{np.percentile(sen_5to10_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr95_list, 2.5):.3f}-{np.percentile(sen_10to15_thr95_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr95_list, 2.5):.3f}-{np.percentile(spe_thr95_list, 97.5):.3f})"
)

95% CI (without gray)
AUROC(0 ≤ t < 5): (0.646-0.699)
∆AUROC (0to5 vs. 5to10): (-0.061-0.010)
AUROC(5 ≤ t < 10): (0.673-0.720)
∆AUROC (5to10 vs. 10to15): (-0.151--0.090)
AUROC(10 ≤ t < 15): (0.800-0.836)
∆AUROC (0to5 vs. 10to15): (0.114-0.178)
Spe80% (train):
- Sen(0 ≤ t < 5):: (0.466-0.550)
- Sen(5 ≤ t < 10):: (0.522-0.600)
- Sen(10 ≤ t < 15):: (0.743-0.794)
-------------------------
- Spe: (0.684-0.734)
Spe90% (train):
- Sen(0 ≤ t < 5):: (0.296-0.379)
- Sen(5 ≤ t < 10):: (0.345-0.422)
- Sen(10 ≤ t < 15):: (0.595-0.654)
-------------------------
- Spe: (0.824-0.858)
Spe95% (train):
- Sen(0 ≤ t < 5):: (0.180-0.249)
- Sen(5 ≤ t < 10):: (0.223-0.293)
- Sen(10 ≤ t < 15):: (0.423-0.485)
-------------------------
- Spe: (0.910-0.931)


# CNN

In [105]:
loaded_model = load_model(model_path / "cnn.model.keras")

In [106]:
prob_y_0to5 = loaded_model.predict(X_0to5_s).reshape(-1)
prob_y_5to10 = loaded_model.predict(X_5to10_s).reshape(-1)
prob_y_10to15 = loaded_model.predict(X_10to15_s).reshape(-1)

595/595 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
606/606 ━━━━━━━━━━━━━━━━━━━━ 1s 989us/step
637/637 ━━━━━━━━━━━━━━━━━━━━ 1s 979us/step


In [107]:
print(f"AUROC(0min ≤ t < 5min): {roc_auc_score(y_0to5, prob_y_0to5):.3f}")
print(f"AUROC(5min ≤ t < 10min): {roc_auc_score(y_5to10, prob_y_5to10):.3f}")
print(f"AUROC(10min ≤ t < 15min): {roc_auc_score(y_10to15, prob_y_10to15):.3f}")

AUROC(0min ≤ t < 5min): 0.657
AUROC(5min ≤ t < 10min): 0.683
AUROC(10min ≤ t < 15min): 0.818


In [108]:
print("Specificity(train): 0.8")
pred_0to5 = (prob_y_0to5 >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.8
Sen(0 ≤ t < 5): 0.457
Sen(5 ≤ t < 10): 0.504
Sen(10 ≤ t < 15): 0.739
-------------------------
Spe: 0.761


In [109]:
print("Specificity(train): 0.9")
pred_0to5 = (prob_y_0to5 >= 0.71).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.71).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.71).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.9
Sen(0 ≤ t < 5): 0.288
Sen(5 ≤ t < 10): 0.336
Sen(10 ≤ t < 15): 0.585
-------------------------
Spe: 0.870


In [110]:
print("Specificity(train): 0.95")
pred_0to5 = (prob_y_0to5 >= 0.79).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.79).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.79).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.95
Sen(0 ≤ t < 5): 0.183
Sen(5 ≤ t < 10): 0.238
Sen(10 ≤ t < 15): 0.455
-------------------------
Spe: 0.927


In [111]:
pat_ids_0to5 = np.unique(cid_0to5)
pat_ids_5to10 = np.unique(cid_5to10)
pat_ids_10to15 = np.unique(cid_10to15)
idx_by_pat_0to5 = {p: np.where(cid_0to5 == p)[0] for p in pat_ids_0to5}
idx_by_pat_5to10 = {p: np.where(cid_5to10 == p)[0] for p in pat_ids_5to10}
idx_by_pat_10to15 = {p: np.where(cid_10to15 == p)[0] for p in pat_ids_10to15}
np.random.seed(config["seed"])  # 42

auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
# 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
delta_list1, delta_list2, delta_list3 = [], [], []
sen_0to5_thr80_list, sen_5to10_thr80_list, sen_10to15_thr80_list = [], [], []
sen_0to5_thr90_list, sen_5to10_thr90_list, sen_10to15_thr90_list = [], [], []
sen_0to5_thr95_list, sen_5to10_thr95_list, sen_10to15_thr95_list = [], [], []
spe_thr80_list, spe_thr90_list, spe_thr95_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_0to5 = score_sampled_auroc(idx_by_pat_0to5, y_0to5, prob_y_0to5)
    auroc_5to10 = score_sampled_auroc(idx_by_pat_5to10, y_5to10, prob_y_5to10)
    auroc_10to15 = score_sampled_auroc(idx_by_pat_10to15, y_10to15, prob_y_10to15)
    auroc_0to5_list.append(auroc_0to5)
    auroc_5to10_list.append(auroc_5to10)
    auroc_10to15_list.append(auroc_10to15)
    delta_list1.append(auroc_0to5 - auroc_5to10)
    delta_list2.append(auroc_5to10 - auroc_10to15)
    delta_list3.append(auroc_10to15 - auroc_0to5)

    # Spe (train): 80%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.58)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.58
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.58
    )
    sen_0to5_thr80_list.append(sen_0to5)
    sen_5to10_thr80_list.append(sen_5to10)
    sen_10to15_thr80_list.append(sen_10to15)
    spe_thr80_list.append(spe_0to5)

    # Spe (train): 90%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.71)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.71
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.71
    )
    sen_0to5_thr90_list.append(sen_0to5)
    sen_5to10_thr90_list.append(sen_5to10)
    sen_10to15_thr90_list.append(sen_10to15)
    spe_thr90_list.append(spe_0to5)

    # Spe (train): 95%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.79)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.79
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.79
    )
    sen_0to5_thr95_list.append(sen_0to5)
    sen_5to10_thr95_list.append(sen_5to10)
    sen_10to15_thr95_list.append(sen_10to15)
    spe_thr95_list.append(spe_0to5)

100%|██████████████████████████████████| 2000/2000 [01:08<00:00, 29.07it/s]


In [112]:
print("95% CI (without gray)")
print(
    f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_0to5_list, 2.5):.3f}-{np.percentile(auroc_0to5_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
)
print(
    f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_5to10_list, 2.5):.3f}-{np.percentile(auroc_5to10_list, 97.5):.3f})"
)
print(
    f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
)
print(
    f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_10to15_list, 2.5):.3f}-{np.percentile(auroc_10to15_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
)

print("Spe80% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr80_list, 2.5):.3f}-{np.percentile(sen_0to5_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr80_list, 2.5):.3f}-{np.percentile(sen_5to10_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr80_list, 2.5):.3f}-{np.percentile(sen_10to15_thr80_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr80_list, 2.5):.3f}-{np.percentile(spe_thr80_list, 97.5):.3f})"
)

print("Spe90% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr90_list, 2.5):.3f}-{np.percentile(sen_0to5_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr90_list, 2.5):.3f}-{np.percentile(sen_5to10_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr90_list, 2.5):.3f}-{np.percentile(sen_10to15_thr90_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr90_list, 2.5):.3f}-{np.percentile(spe_thr90_list, 97.5):.3f})"
)

print("Spe95% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr95_list, 2.5):.3f}-{np.percentile(sen_0to5_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr95_list, 2.5):.3f}-{np.percentile(sen_5to10_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr95_list, 2.5):.3f}-{np.percentile(sen_10to15_thr95_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr95_list, 2.5):.3f}-{np.percentile(spe_thr95_list, 97.5):.3f})"
)

95% CI (without gray)
AUROC(0 ≤ t < 5): (0.627-0.686)
∆AUROC (0to5 vs. 5to10): (-0.066-0.010)
AUROC(5 ≤ t < 10): (0.656-0.709)
∆AUROC (5to10 vs. 10to15): (-0.167--0.102)
AUROC(10 ≤ t < 15): (0.799-0.835)
∆AUROC (0to5 vs. 10to15): (0.128-0.195)
Spe80% (train):
- Sen(0 ≤ t < 5):: (0.414-0.502)
- Sen(5 ≤ t < 10):: (0.463-0.547)
- Sen(10 ≤ t < 15):: (0.711-0.763)
-------------------------
- Spe: (0.735-0.784)
Spe90% (train):
- Sen(0 ≤ t < 5):: (0.247-0.329)
- Sen(5 ≤ t < 10):: (0.295-0.377)
- Sen(10 ≤ t < 15):: (0.556-0.616)
-------------------------
- Spe: (0.853-0.887)
Spe95% (train):
- Sen(0 ≤ t < 5):: (0.149-0.218)
- Sen(5 ≤ t < 10):: (0.202-0.276)
- Sen(10 ≤ t < 15):: (0.423-0.484)
-------------------------
- Spe: (0.914-0.938)


# ResNet

In [113]:
loaded_model = load_model(model_path / "resnet.model.keras")

In [114]:
prob_y_0to5 = loaded_model.predict(X_0to5_s).reshape(-1)
prob_y_5to10 = loaded_model.predict(X_5to10_s).reshape(-1)
prob_y_10to15 = loaded_model.predict(X_10to15_s).reshape(-1)

595/595 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step
606/606 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step
637/637 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step


In [115]:
print(f"AUROC(0min ≤ t < 5min): {roc_auc_score(y_0to5, prob_y_0to5):.3f}")
print(f"AUROC(5min ≤ t < 10min): {roc_auc_score(y_5to10, prob_y_5to10):.3f}")
print(f"AUROC(10min ≤ t < 15min): {roc_auc_score(y_10to15, prob_y_10to15):.3f}")

AUROC(0min ≤ t < 5min): 0.671
AUROC(5min ≤ t < 10min): 0.696
AUROC(10min ≤ t < 15min): 0.811


In [116]:
print("Specificity(train): 0.8")
pred_0to5 = (prob_y_0to5 >= 0.74).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.74).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.74).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.8
Sen(0 ≤ t < 5): 0.672
Sen(5 ≤ t < 10): 0.702
Sen(10 ≤ t < 15): 0.868
-------------------------
Spe: 0.566


In [117]:
print("Specificity(train): 0.9")
pred_0to5 = (prob_y_0to5 >= 0.85).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.85).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.85).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.9
Sen(0 ≤ t < 5): 0.435
Sen(5 ≤ t < 10): 0.484
Sen(10 ≤ t < 15): 0.701
-------------------------
Spe: 0.772


In [118]:
print("Specificity(train): 0.95")
pred_0to5 = (prob_y_0to5 >= 0.91).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.91).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.91).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.95
Sen(0 ≤ t < 5): 0.269
Sen(5 ≤ t < 10): 0.322
Sen(10 ≤ t < 15): 0.539
-------------------------
Spe: 0.885


In [119]:
pat_ids_0to5 = np.unique(cid_0to5)
pat_ids_5to10 = np.unique(cid_5to10)
pat_ids_10to15 = np.unique(cid_10to15)
idx_by_pat_0to5 = {p: np.where(cid_0to5 == p)[0] for p in pat_ids_0to5}
idx_by_pat_5to10 = {p: np.where(cid_5to10 == p)[0] for p in pat_ids_5to10}
idx_by_pat_10to15 = {p: np.where(cid_10to15 == p)[0] for p in pat_ids_10to15}
np.random.seed(config["seed"])  # 42

auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
# 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
delta_list1, delta_list2, delta_list3 = [], [], []
sen_0to5_thr80_list, sen_5to10_thr80_list, sen_10to15_thr80_list = [], [], []
sen_0to5_thr90_list, sen_5to10_thr90_list, sen_10to15_thr90_list = [], [], []
sen_0to5_thr95_list, sen_5to10_thr95_list, sen_10to15_thr95_list = [], [], []
spe_thr80_list, spe_thr90_list, spe_thr95_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_0to5 = score_sampled_auroc(idx_by_pat_0to5, y_0to5, prob_y_0to5)
    auroc_5to10 = score_sampled_auroc(idx_by_pat_5to10, y_5to10, prob_y_5to10)
    auroc_10to15 = score_sampled_auroc(idx_by_pat_10to15, y_10to15, prob_y_10to15)
    auroc_0to5_list.append(auroc_0to5)
    auroc_5to10_list.append(auroc_5to10)
    auroc_10to15_list.append(auroc_10to15)
    delta_list1.append(auroc_0to5 - auroc_5to10)
    delta_list2.append(auroc_5to10 - auroc_10to15)
    delta_list3.append(auroc_10to15 - auroc_0to5)

    # Spe (train): 80%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.74)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.74
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.74
    )
    sen_0to5_thr80_list.append(sen_0to5)
    sen_5to10_thr80_list.append(sen_5to10)
    sen_10to15_thr80_list.append(sen_10to15)
    spe_thr80_list.append(spe_0to5)

    # Spe (train): 90%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.85)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.85
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.85
    )
    sen_0to5_thr90_list.append(sen_0to5)
    sen_5to10_thr90_list.append(sen_5to10)
    sen_10to15_thr90_list.append(sen_10to15)
    spe_thr90_list.append(spe_0to5)

    # Spe (train): 95%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.91)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.91
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.91
    )
    sen_0to5_thr95_list.append(sen_0to5)
    sen_5to10_thr95_list.append(sen_5to10)
    sen_10to15_thr95_list.append(sen_10to15)
    spe_thr95_list.append(spe_0to5)

100%|██████████████████████████████████| 2000/2000 [01:10<00:00, 28.23it/s]


In [120]:
print("95% CI (without gray)")
print(
    f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_0to5_list, 2.5):.3f}-{np.percentile(auroc_0to5_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
)
print(
    f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_5to10_list, 2.5):.3f}-{np.percentile(auroc_5to10_list, 97.5):.3f})"
)
print(
    f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
)
print(
    f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_10to15_list, 2.5):.3f}-{np.percentile(auroc_10to15_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
)

print("Spe80% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr80_list, 2.5):.3f}-{np.percentile(sen_0to5_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr80_list, 2.5):.3f}-{np.percentile(sen_5to10_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr80_list, 2.5):.3f}-{np.percentile(sen_10to15_thr80_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr80_list, 2.5):.3f}-{np.percentile(spe_thr80_list, 97.5):.3f})"
)

print("Spe90% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr90_list, 2.5):.3f}-{np.percentile(sen_0to5_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr90_list, 2.5):.3f}-{np.percentile(sen_5to10_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr90_list, 2.5):.3f}-{np.percentile(sen_10to15_thr90_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr90_list, 2.5):.3f}-{np.percentile(spe_thr90_list, 97.5):.3f})"
)

print("Spe95% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr95_list, 2.5):.3f}-{np.percentile(sen_0to5_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr95_list, 2.5):.3f}-{np.percentile(sen_5to10_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr95_list, 2.5):.3f}-{np.percentile(sen_10to15_thr95_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr95_list, 2.5):.3f}-{np.percentile(spe_thr95_list, 97.5):.3f})"
)

95% CI (without gray)
AUROC(0 ≤ t < 5): (0.642-0.700)
∆AUROC (0to5 vs. 5to10): (-0.065-0.012)
AUROC(5 ≤ t < 10): (0.669-0.722)
∆AUROC (5to10 vs. 10to15): (-0.149--0.081)
AUROC(10 ≤ t < 15): (0.792-0.830)
∆AUROC (0to5 vs. 10to15): (0.105-0.175)
Spe80% (train):
- Sen(0 ≤ t < 5):: (0.630-0.713)
- Sen(5 ≤ t < 10):: (0.661-0.741)
- Sen(10 ≤ t < 15):: (0.847-0.889)
-------------------------
- Spe: (0.530-0.600)
Spe90% (train):
- Sen(0 ≤ t < 5):: (0.388-0.481)
- Sen(5 ≤ t < 10):: (0.440-0.526)
- Sen(10 ≤ t < 15):: (0.671-0.731)
-------------------------
- Spe: (0.747-0.796)
Spe95% (train):
- Sen(0 ≤ t < 5):: (0.230-0.308)
- Sen(5 ≤ t < 10):: (0.281-0.362)
- Sen(10 ≤ t < 15):: (0.506-0.571)
-------------------------
- Spe: (0.868-0.899)


# LSTMFCN

In [121]:
loaded_model = load_model(model_path / "lstmfcn.model.keras")

In [122]:
prob_y_0to5 = loaded_model.predict(X_0to5_s).reshape(-1)
prob_y_5to10 = loaded_model.predict(X_5to10_s).reshape(-1)
prob_y_10to15 = loaded_model.predict(X_10to15_s).reshape(-1)

595/595 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step
606/606 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step
637/637 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step


In [123]:
print(f"AUROC(0min ≤ t < 5min): {roc_auc_score(y_0to5, prob_y_0to5):.3f}")
print(f"AUROC(5min ≤ t < 10min): {roc_auc_score(y_5to10, prob_y_5to10):.3f}")
print(f"AUROC(10min ≤ t < 15min): {roc_auc_score(y_10to15, prob_y_10to15):.3f}")

AUROC(0min ≤ t < 5min): 0.654
AUROC(5min ≤ t < 10min): 0.677
AUROC(10min ≤ t < 15min): 0.799


In [124]:
print("Specificity(train): 0.8")
pred_0to5 = (prob_y_0to5 >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.8
Sen(0 ≤ t < 5): 0.713
Sen(5 ≤ t < 10): 0.746
Sen(10 ≤ t < 15): 0.877
-------------------------
Spe: 0.483


In [125]:
print("Specificity(train): 0.9")
pred_0to5 = (prob_y_0to5 >= 0.72).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.72).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.72).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.9
Sen(0 ≤ t < 5): 0.501
Sen(5 ≤ t < 10): 0.529
Sen(10 ≤ t < 15): 0.745
-------------------------
Spe: 0.715


In [126]:
print("Specificity(train): 0.95")
pred_0to5 = (prob_y_0to5 >= 0.80).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.80).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.80).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.95
Sen(0 ≤ t < 5): 0.356
Sen(5 ≤ t < 10): 0.391
Sen(10 ≤ t < 15): 0.614
-------------------------
Spe: 0.831


In [127]:
pat_ids_0to5 = np.unique(cid_0to5)
pat_ids_5to10 = np.unique(cid_5to10)
pat_ids_10to15 = np.unique(cid_10to15)
idx_by_pat_0to5 = {p: np.where(cid_0to5 == p)[0] for p in pat_ids_0to5}
idx_by_pat_5to10 = {p: np.where(cid_5to10 == p)[0] for p in pat_ids_5to10}
idx_by_pat_10to15 = {p: np.where(cid_10to15 == p)[0] for p in pat_ids_10to15}
np.random.seed(config["seed"])  # 42

auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
# 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
delta_list1, delta_list2, delta_list3 = [], [], []
sen_0to5_thr80_list, sen_5to10_thr80_list, sen_10to15_thr80_list = [], [], []
sen_0to5_thr90_list, sen_5to10_thr90_list, sen_10to15_thr90_list = [], [], []
sen_0to5_thr95_list, sen_5to10_thr95_list, sen_10to15_thr95_list = [], [], []
spe_thr80_list, spe_thr90_list, spe_thr95_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_0to5 = score_sampled_auroc(idx_by_pat_0to5, y_0to5, prob_y_0to5)
    auroc_5to10 = score_sampled_auroc(idx_by_pat_5to10, y_5to10, prob_y_5to10)
    auroc_10to15 = score_sampled_auroc(idx_by_pat_10to15, y_10to15, prob_y_10to15)
    auroc_0to5_list.append(auroc_0to5)
    auroc_5to10_list.append(auroc_5to10)
    auroc_10to15_list.append(auroc_10to15)
    delta_list1.append(auroc_0to5 - auroc_5to10)
    delta_list2.append(auroc_5to10 - auroc_10to15)
    delta_list3.append(auroc_10to15 - auroc_0to5)

    # Spe (train): 80%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.58)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.58
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.58
    )
    sen_0to5_thr80_list.append(sen_0to5)
    sen_5to10_thr80_list.append(sen_5to10)
    sen_10to15_thr80_list.append(sen_10to15)
    spe_thr80_list.append(spe_0to5)

    # Spe (train): 90%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.72)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.72
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.72
    )
    sen_0to5_thr90_list.append(sen_0to5)
    sen_5to10_thr90_list.append(sen_5to10)
    sen_10to15_thr90_list.append(sen_10to15)
    spe_thr90_list.append(spe_0to5)

    # Spe (train): 95%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.80)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.80
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.80
    )
    sen_0to5_thr95_list.append(sen_0to5)
    sen_5to10_thr95_list.append(sen_5to10)
    sen_10to15_thr95_list.append(sen_10to15)
    spe_thr95_list.append(spe_0to5)

100%|██████████████████████████████████| 2000/2000 [01:10<00:00, 28.50it/s]


In [128]:
print("95% CI (without gray)")
print(
    f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_0to5_list, 2.5):.3f}-{np.percentile(auroc_0to5_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
)
print(
    f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_5to10_list, 2.5):.3f}-{np.percentile(auroc_5to10_list, 97.5):.3f})"
)
print(
    f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
)
print(
    f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_10to15_list, 2.5):.3f}-{np.percentile(auroc_10to15_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
)

print("Spe80% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr80_list, 2.5):.3f}-{np.percentile(sen_0to5_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr80_list, 2.5):.3f}-{np.percentile(sen_5to10_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr80_list, 2.5):.3f}-{np.percentile(sen_10to15_thr80_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr80_list, 2.5):.3f}-{np.percentile(spe_thr80_list, 97.5):.3f})"
)

print("Spe90% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr90_list, 2.5):.3f}-{np.percentile(sen_0to5_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr90_list, 2.5):.3f}-{np.percentile(sen_5to10_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr90_list, 2.5):.3f}-{np.percentile(sen_10to15_thr90_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr90_list, 2.5):.3f}-{np.percentile(spe_thr90_list, 97.5):.3f})"
)

print("Spe95% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr95_list, 2.5):.3f}-{np.percentile(sen_0to5_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr95_list, 2.5):.3f}-{np.percentile(sen_5to10_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr95_list, 2.5):.3f}-{np.percentile(sen_10to15_thr95_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr95_list, 2.5):.3f}-{np.percentile(spe_thr95_list, 97.5):.3f})"
)

95% CI (without gray)
AUROC(0 ≤ t < 5): (0.624-0.684)
∆AUROC (0to5 vs. 5to10): (-0.064-0.020)
AUROC(5 ≤ t < 10): (0.649-0.705)
∆AUROC (5to10 vs. 10to15): (-0.157--0.086)
AUROC(10 ≤ t < 15): (0.778-0.820)
∆AUROC (0to5 vs. 10to15): (0.107-0.182)
Spe80% (train):
- Sen(0 ≤ t < 5):: (0.670-0.753)
- Sen(5 ≤ t < 10):: (0.705-0.782)
- Sen(10 ≤ t < 15):: (0.855-0.898)
-------------------------
- Spe: (0.447-0.517)
Spe90% (train):
- Sen(0 ≤ t < 5):: (0.454-0.550)
- Sen(5 ≤ t < 10):: (0.485-0.571)
- Sen(10 ≤ t < 15):: (0.714-0.774)
-------------------------
- Spe: (0.689-0.741)
Spe95% (train):
- Sen(0 ≤ t < 5):: (0.312-0.401)
- Sen(5 ≤ t < 10):: (0.345-0.436)
- Sen(10 ≤ t < 15):: (0.579-0.644)
-------------------------
- Spe: (0.811-0.851)


# InceptionTime

In [129]:
loaded_model = load_model(model_path / "inceptiontime.model.keras")

In [130]:
prob_y_0to5 = loaded_model.predict(X_0to5_s).reshape(-1)
prob_y_5to10 = loaded_model.predict(X_5to10_s).reshape(-1)
prob_y_10to15 = loaded_model.predict(X_10to15_s).reshape(-1)

595/595 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step
606/606 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step
637/637 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step


In [131]:
print(f"AUROC(0min ≤ t < 5min): {roc_auc_score(y_0to5, prob_y_0to5):.3f}")
print(f"AUROC(5min ≤ t < 10min): {roc_auc_score(y_5to10, prob_y_5to10):.3f}")
print(f"AUROC(10min ≤ t < 15min): {roc_auc_score(y_10to15, prob_y_10to15):.3f}")

AUROC(0min ≤ t < 5min): 0.667
AUROC(5min ≤ t < 10min): 0.689
AUROC(10min ≤ t < 15min): 0.791


In [132]:
print("Specificity(train): 0.8")
pred_0to5 = (prob_y_0to5 >= 0.65).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.65).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.65).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.8
Sen(0 ≤ t < 5): 0.686
Sen(5 ≤ t < 10): 0.704
Sen(10 ≤ t < 15): 0.843
-------------------------
Spe: 0.563


In [133]:
print("Specificity(train): 0.9")
pred_0to5 = (prob_y_0to5 >= 0.79).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.79).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.79).astype(int)  # Spe (train): 90%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.9
Sen(0 ≤ t < 5): 0.472
Sen(5 ≤ t < 10): 0.507
Sen(10 ≤ t < 15): 0.699
-------------------------
Spe: 0.743


In [134]:
print("Specificity(train): 0.95")
pred_0to5 = (prob_y_0to5 >= 0.86).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_0to5, pred_0to5).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(0 ≤ t < 5): {sen:.3f}")
pred_5to10 = (prob_y_5to10 >= 0.86).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_5to10, pred_5to10).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(5 ≤ t < 10): {sen:.3f}")
pred_10to15 = (prob_y_10to15 >= 0.86).astype(int)  # Spe (train): 95%
tn, fp, fn, tp = confusion_matrix(y_10to15, pred_10to15).ravel()
sen = tp / (tp + fn)
spe = tn / (tn + fp)
print(f"Sen(10 ≤ t < 15): {sen:.3f}")
print("-----" * 5)
print(f"Spe: {spe:.3f}")

Specificity(train): 0.95
Sen(0 ≤ t < 5): 0.302
Sen(5 ≤ t < 10): 0.368
Sen(10 ≤ t < 15): 0.570
-------------------------
Spe: 0.841


In [135]:
pat_ids_0to5 = np.unique(cid_0to5)
pat_ids_5to10 = np.unique(cid_5to10)
pat_ids_10to15 = np.unique(cid_10to15)
idx_by_pat_0to5 = {p: np.where(cid_0to5 == p)[0] for p in pat_ids_0to5}
idx_by_pat_5to10 = {p: np.where(cid_5to10 == p)[0] for p in pat_ids_5to10}
idx_by_pat_10to15 = {p: np.where(cid_10to15 == p)[0] for p in pat_ids_10to15}
np.random.seed(config["seed"])  # 42

auroc_0to5_list, auroc_5to10_list, auroc_10to15_list = [], [], []
# 0to5 vs. 5to10 / 5to10 vs. 10to15 / 0to5 vs. 10to15
delta_list1, delta_list2, delta_list3 = [], [], []
sen_0to5_thr80_list, sen_5to10_thr80_list, sen_10to15_thr80_list = [], [], []
sen_0to5_thr90_list, sen_5to10_thr90_list, sen_10to15_thr90_list = [], [], []
sen_0to5_thr95_list, sen_5to10_thr95_list, sen_10to15_thr95_list = [], [], []
spe_thr80_list, spe_thr90_list, spe_thr95_list = [], [], []
n = 2000
for i in tqdm(range(n), ncols=75):
    auroc_0to5 = score_sampled_auroc(idx_by_pat_0to5, y_0to5, prob_y_0to5)
    auroc_5to10 = score_sampled_auroc(idx_by_pat_5to10, y_5to10, prob_y_5to10)
    auroc_10to15 = score_sampled_auroc(idx_by_pat_10to15, y_10to15, prob_y_10to15)
    auroc_0to5_list.append(auroc_0to5)
    auroc_5to10_list.append(auroc_5to10)
    auroc_10to15_list.append(auroc_10to15)
    delta_list1.append(auroc_0to5 - auroc_5to10)
    delta_list2.append(auroc_5to10 - auroc_10to15)
    delta_list3.append(auroc_10to15 - auroc_0to5)

    # Spe (train): 80%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.65)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.65
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.65
    )
    sen_0to5_thr80_list.append(sen_0to5)
    sen_5to10_thr80_list.append(sen_5to10)
    sen_10to15_thr80_list.append(sen_10to15)
    spe_thr80_list.append(spe_0to5)

    # Spe (train): 90%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.79)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.79
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.79
    )
    sen_0to5_thr90_list.append(sen_0to5)
    sen_5to10_thr90_list.append(sen_5to10)
    sen_10to15_thr90_list.append(sen_10to15)
    spe_thr90_list.append(spe_0to5)

    # Spe (train): 95%
    sen_0to5, spe_0to5 = eval_sampled_clf(idx_by_pat_0to5, y_0to5, prob_y_0to5, 0.86)
    sen_5to10, spe_5to10 = eval_sampled_clf(
        idx_by_pat_5to10, y_5to10, prob_y_5to10, 0.86
    )
    sen_10to15, spe_10to15 = eval_sampled_clf(
        idx_by_pat_10to15, y_10to15, prob_y_10to15, 0.86
    )
    sen_0to5_thr95_list.append(sen_0to5)
    sen_5to10_thr95_list.append(sen_5to10)
    sen_10to15_thr95_list.append(sen_10to15)
    spe_thr95_list.append(spe_0to5)

100%|██████████████████████████████████| 2000/2000 [01:10<00:00, 28.53it/s]


In [136]:
print("95% CI (without gray)")
print(
    f"AUROC(0 ≤ t < 5): ({np.percentile(auroc_0to5_list, 2.5):.3f}-{np.percentile(auroc_0to5_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 5to10): ({np.percentile(delta_list1, 2.5):.3f}-{np.percentile(delta_list1, 97.5):.3f})"
)
print(
    f"AUROC(5 ≤ t < 10): ({np.percentile(auroc_5to10_list, 2.5):.3f}-{np.percentile(auroc_5to10_list, 97.5):.3f})"
)
print(
    f"∆AUROC (5to10 vs. 10to15): ({np.percentile(delta_list2, 2.5):.3f}-{np.percentile(delta_list2, 97.5):.3f})"
)
print(
    f"AUROC(10 ≤ t < 15): ({np.percentile(auroc_10to15_list, 2.5):.3f}-{np.percentile(auroc_10to15_list, 97.5):.3f})"
)
print(
    f"∆AUROC (0to5 vs. 10to15): ({np.percentile(delta_list3, 2.5):.3f}-{np.percentile(delta_list3, 97.5):.3f})"
)

print("Spe80% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr80_list, 2.5):.3f}-{np.percentile(sen_0to5_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr80_list, 2.5):.3f}-{np.percentile(sen_5to10_thr80_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr80_list, 2.5):.3f}-{np.percentile(sen_10to15_thr80_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr80_list, 2.5):.3f}-{np.percentile(spe_thr80_list, 97.5):.3f})"
)

print("Spe90% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr90_list, 2.5):.3f}-{np.percentile(sen_0to5_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr90_list, 2.5):.3f}-{np.percentile(sen_5to10_thr90_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr90_list, 2.5):.3f}-{np.percentile(sen_10to15_thr90_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr90_list, 2.5):.3f}-{np.percentile(spe_thr90_list, 97.5):.3f})"
)

print("Spe95% (train):")
print(
    f"- Sen(0 ≤ t < 5):: ({np.percentile(sen_0to5_thr95_list, 2.5):.3f}-{np.percentile(sen_0to5_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(5 ≤ t < 10):: ({np.percentile(sen_5to10_thr95_list, 2.5):.3f}-{np.percentile(sen_5to10_thr95_list, 97.5):.3f})"
)
print(
    f"- Sen(10 ≤ t < 15):: ({np.percentile(sen_10to15_thr95_list, 2.5):.3f}-{np.percentile(sen_10to15_thr95_list, 97.5):.3f})"
)
print("-----" * 5)
print(
    f"- Spe: ({np.percentile(spe_thr95_list, 2.5):.3f}-{np.percentile(spe_thr95_list, 97.5):.3f})"
)

95% CI (without gray)
AUROC(0 ≤ t < 5): (0.635-0.696)
∆AUROC (0to5 vs. 5to10): (-0.065-0.018)
AUROC(5 ≤ t < 10): (0.659-0.719)
∆AUROC (5to10 vs. 10to15): (-0.138--0.063)
AUROC(10 ≤ t < 15): (0.767-0.813)
∆AUROC (0to5 vs. 10to15): (0.087-0.163)
Spe80% (train):
- Sen(0 ≤ t < 5):: (0.642-0.728)
- Sen(5 ≤ t < 10):: (0.662-0.747)
- Sen(10 ≤ t < 15):: (0.818-0.866)
-------------------------
- Spe: (0.525-0.600)
Spe90% (train):
- Sen(0 ≤ t < 5):: (0.421-0.521)
- Sen(5 ≤ t < 10):: (0.457-0.556)
- Sen(10 ≤ t < 15):: (0.666-0.732)
-------------------------
- Spe: (0.712-0.772)
Spe95% (train):
- Sen(0 ≤ t < 5):: (0.259-0.346)
- Sen(5 ≤ t < 10):: (0.323-0.414)
- Sen(10 ≤ t < 15):: (0.531-0.607)
-------------------------
- Spe: (0.815-0.864)
